# Trabalho 1 - ANADI

## Manipulação de Dados

## Processamento da Iluminação Publica (IP_data)

In [ ]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy import stats
from pathlib import Path
from scipy.stats import shapiro
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown

In [ ]:
# Configurar diretório base do projeto
notebook_dir = Path.cwd()
if notebook_dir.name == "src": os.chdir(notebook_dir.parent)

print(f"Diretório de trabalho: {Path.cwd()}")

In [ ]:
# Importar dados do Excel
data = pd.read_excel('data/IP_data.xlsx')

print('Dados importados:')
print(data)

## Criar Variáveis

In [ ]:
# 1 se tipo de lâmpada é Sódio ou Mercúrio, 0 for outro tipo
data['Is_Ineficiente'] = data['Tipo de Lâmpada'].isin(['Sódio', 'Mercúrio']).astype(int)

print(f"Distribuição Is_Inefiente:")
print(data['Is_Ineficiente'].value_counts())

In [ ]:
# Criar variável Potência kW
# Concertendo potência de W para kW

data['Potencia_kw'] = data['Potência Instalada Total (W)'] / 1000

print(f"\nEstatísticas Potencia_kW: ")
print(data['Potencia_kw'].describe())

## Agregação por Concelho

In [ ]:
# Agrupar por CodDistritoConcelho

agrupado = data.groupby('CodDistritoConcelho').agg(
    P_IP_TOTAL = ('Potencia_kw', 'sum'),
    P_IP_Inef = ('Potencia_kw', lambda x: x[data.loc[x.index, 'Is_Ineficiente'] == 1].sum())
).reset_index()

print("\n Agregação por Concelho:")
print(agrupado)

In [ ]:
# Mostrar informações adicionais

print(f"\nTotal de concelhos: {len(agrupado)}")

print(f"P_IP_TOTAL (soma global): {agrupado['P_IP_TOTAL'].sum():.2f} kW")

print(f"P_IP_Inef (soma global): {agrupado['P_IP_Inef'].sum():.2f} kW")

print(f"% Potência Ineficiente: {(agrupado['P_IP_Inef'].sum() / agrupado['P_IP_TOTAL'].sum() * 100):.2f}%")


## Processamento dos Postos de Transformação (PTD data)

In [ ]:
# Importar dados do Excel
ptd_data = pd.read_excel('data/PTD_data.xlsx')

print('Dados importados:')
print(data)

In [ ]:
# Converter nível de utilização para decimal

# Converter 'Nível de Utilização [%]' de formato '60%-79%' para valor decimal (ex: 0.79)
# Extrai o último número antes do '%' final

ptd_data['Utilização Decimal'] = (
    ptd_data['Nível de Utilização [%]']
    .astype(str)
    .str.extract(r'(\d+)%$')[0] # Extrai o número antes do último '%'
    .astype(float) / 100
)

print("\nConversão Nível de Utilização:")
print(ptd_data[['Nível de Utilização [%]', 'Utilização Decimal']].drop_duplicates().sort_values('Utilização Decimal'))



## Agrupar por CodDistritoConcelho

In [ ]:
agrupado_ptd = ptd_data.groupby('CodDistritoConcelho').agg(
    Cap_PTD = ('Potência instalada [kVA]', 'sum'),
    Util_Media = ('Utilização Decimal', 'mean'),
    N_PTDs = ('Código de Instalação', 'count')
).reset_index()

print("\nAgregação PTD por Concelho")
print(agrupado_ptd)

In [ ]:
# Mostrar informações adicionais

print(f"\nTotal de concelhos com PTD: {len(agrupado_ptd)}")
print(f"Cap_PTD total: {agrupado_ptd['Cap_PTD'].sum():.2f} kVA")
print(f"Utilização média global: {agrupado_ptd['Util_Media'].mean():.2%}")
print(f"Total de PTDs: {agrupado_ptd['N_PTDs'].sum()}")

print("\nEstatísticas dos PTDs por concelho:")
print(agrupado_ptd.describe())

# Criar Dataset Final

In [ ]:
# Juntar agregação da IP com agregação dos PTDs
df_final = pd.merge(
    agrupado,
    agrupado_ptd,
    on="CodDistritoConcelho",
    how="inner"
)

# Recuperar informação descritiva dos municípios
info_municipios = data[["CodDistritoConcelho", "Distrito", "Concelho"]].drop_duplicates()

# Juntar distrito e concelho ao dataset final
df_final = pd.merge(
    df_final,
    info_municipios,
    on="CodDistritoConcelho",
    how="left"
)

# =========================================
# Criar variáveis derivadas do enunciado
# =========================================

# Ganho LED: potência libertada pela substituição de lâmpadas ineficientes
df_final["Ganho_LED"] = df_final["P_IP_Inef"] * 0.65

# Folga da rede: capacidade disponível com margem de segurança de 92%
df_final["PFolga"] = (df_final["Cap_PTD"] * 0.92) * (1 - df_final["Util_Media"])

# Carga VE: carga necessária para carregadores de 22 kW em todos os PTDs
df_final["PVE"] = df_final["N_PTDs"] * 22 * 0.60

# Saldo final de viabilidade
df_final["D"] = df_final["PFolga"] + df_final["Ganho_LED"] - df_final["PVE"]

# Rate de ineficiência
df_final["Rate_Ineficiencia"] = df_final["P_IP_Inef"] / df_final["P_IP_TOTAL"]

# Tratar valores inválidos
df_final["Rate_Ineficiencia"] = df_final["Rate_Ineficiencia"].replace([float("inf"), -float("inf")], pd.NA)
df_final["Rate_Ineficiencia"] = df_final["Rate_Ineficiencia"].fillna(0)

# =========================================
# Reorganizar colunas
# =========================================

df_final = df_final[
    [
        "Distrito",
        "Concelho",
        "CodDistritoConcelho",
        "P_IP_TOTAL",
        "P_IP_Inef",
        "Rate_Ineficiencia",
        "Cap_PTD",
        "Util_Media",
        "N_PTDs",
        "Ganho_LED",
        "PFolga",
        "PVE",
        "D"
    ]
]

# =========================================
# Visualização do dataset final
# =========================================

print("Preview do dataset final consolidado:\n")
display(df_final.head())

# =========================================
# Estatísticas globais do dataset
# =========================================

summary_stats = pd.DataFrame({
    "Indicador": [
        "Número total de concelhos",
        "Potência total de iluminação pública (kW)",
        "Potência ineficiente total (kW)",
        "Capacidade total dos PTDs (kVA)",
        "Número total de PTDs"
    ],
    "Valor": [
        len(df_final),
        df_final["P_IP_TOTAL"].sum(),
        df_final["P_IP_Inef"].sum(),
        df_final["Cap_PTD"].sum(),
        df_final["N_PTDs"].sum()
    ]
})

display(summary_stats)

# =========================================
# Guardar dataset final
# =========================================

df_final.to_csv("data/dataset_final.csv", index=False)

print("\nDataset final guardado em: data/dataset_final.csv")

# 4.3. Análise e Exploração de Dados

## Mix tecnológico da iluminação pública

Nesta secção analisa-se a distribuição da potência instalada de iluminação pública entre tecnologia eficiente (LED) e tecnologia convencional.

- Potência convencional: potência associada a luminárias classificadas como ineficientes (`P_IP_Inef`), correspondendo a tecnologias Sódio ou Mercúrio.
- Potência LED: diferença entre a potência total instalada (`P_IP_TOTAL`) e a potência ineficiente.

Objetivos da análise:
- visualizar o peso relativo da tecnologia LED face à convencional;
- identificar se a potência ineficiente se concentra num grupo restrito de municípios.

Esta etapa corresponde a uma análise de estatística descritiva, recorrendo a representações gráficas para comparar categorias tecnológicas e identificar padrões de distribuição da potência instalada.

In [ ]:
# =========================================
# Cálculo da potência LED
# =========================================

df_final["P_IP_LED"] = df_final["P_IP_TOTAL"] - df_final["P_IP_Inef"]

display(df_final[["Concelho","P_IP_TOTAL","P_IP_Inef","P_IP_LED"]].head())

# =========================================
# Totais globais
# =========================================

total_led = df_final["P_IP_LED"].sum()
total_conv = df_final["P_IP_Inef"].sum()
total_ip = df_final["P_IP_TOTAL"].sum()

perc_led = total_led / total_ip
perc_conv = total_conv / total_ip

print("Distribuição global da potência de iluminação pública:\n")

print(f"Potência total LED: {total_led:,.2f} kW ({perc_led:.2%})")
print(f"Potência total Convencional: {total_conv:,.2f} kW ({perc_conv:.2%})")
print(f"Potência total instalada: {total_ip:,.2f} kW")

In [ ]:
labels = ['LED', 'Convencional']
values = [total_led, total_conv]

plt.figure(figsize=(7, 7))

wedges, texts, autotexts = plt.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90
)

# 🔹 labels (fora) → branco
for text in texts:
    text.set_color('white')

# 🔹 percentagens (dentro) → preto
for autotext in autotexts:
    autotext.set_color('black')

plt.title("Mix Tecnológico da Iluminação Pública", color='white')

plt.show()

### Concentração da potência ineficiente por município

Para avaliar se a potência ineficiente se encontra concentrada num grupo restrito de municípios, analisam-se os concelhos com maior valor de `P_IP_Inef`.

Esta análise permite identificar os municípios onde a substituição de tecnologia ineficiente poderá gerar maiores ganhos potenciais de eficiência energética e maior libertação de potência.

In [ ]:
# Ordenar municípios por potência ineficiente
top_municipios = df_final.sort_values("P_IP_Inef", ascending=False).head(10)

# Calcular percentagens de concentração
top10_share = top_municipios["P_IP_Inef"].sum() / df_final["P_IP_Inef"].sum()
top20_share = (
        df_final.sort_values("P_IP_Inef", ascending=False)
        .head(20)["P_IP_Inef"]
        .sum() / df_final["P_IP_Inef"].sum()
)

print("Concentração da potência ineficiente:")
print(f"• Top 10 municípios: {top10_share:.2%} da potência ineficiente total")
print(f"• Top 20 municípios: {top20_share:.2%} da potência ineficiente total")

# Gráfico de barras
plt.figure(figsize=(10,5))

plt.bar(
    top_municipios["Concelho"],
    top_municipios["P_IP_Inef"]
)

plt.title("Top 10 municípios com maior potência ineficiente")
plt.xlabel("Município")
plt.ylabel("Potência ineficiente (kW)")

plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### Interpretação

O gráfico evidencia que a potência associada a luminárias ineficientes se encontra concentrada num número relativamente reduzido de municípios.

O município de **Cascais** apresenta o valor mais elevado de potência ineficiente, seguido por **Sintra, Oeiras, Matosinhos e Almada**, indicando uma maior presença de tecnologia convencional nestas zonas.

Verifica-se ainda que os **10 municípios com maior potência ineficiente concentram cerca de 35.78% da potência ineficiente total**, enquanto os **20 principais representam aproximadamente 54.14%**.

Estes resultados sugerem que a substituição da iluminação convencional por tecnologia **LED** nestes municípios poderá gerar ganhos significativos de eficiência energética e libertação de capacidade na rede elétrica.

## Comparação da utilização dos PTDs por distrito

Nesta secção comparam-se as distribuições do **nível médio de utilização dos Postos de Transformação de Distribuição (PTD)** entre os distritos de Lisboa, Porto, Aveiro e Setúbal.

Para essa comparação utiliza-se um **boxplot**, uma representação gráfica que permite analisar simultaneamente:

- mediana
- quartis
- amplitude interquartil
- presença de valores extremos (outliers)

Esta visualização permite avaliar diferenças na variabilidade da carga da rede entre os distritos analisados e identificar qual apresenta maior dispersão nos níveis de utilização dos PTDs.

In [ ]:
# Distritos a analisar
ordem = ["Lisboa", "Porto", "Aveiro", "Setúbal"]

# Filtrar apenas os distritos pedidos
df_boxplot = df_final[df_final["Distrito"].isin(ordem)].copy()

# Criar listas de dados por distrito
dados = [
    df_boxplot[df_boxplot["Distrito"] == distrito]["Util_Media"]
    for distrito in ordem
]

# Gráfico boxplot simples
plt.figure(figsize=(9,6))

plt.boxplot(
    dados,
    tick_labels=ordem,
    patch_artist=True
)

plt.title("Distribuição da utilização média dos PTDs por distrito")
plt.xlabel("Distrito")
plt.ylabel("Utilização média dos PTDs")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

# Calcular variância
variancia = (
    df_boxplot.groupby("Distrito")["Util_Media"]
    .var()
    .reindex(ordem)
    .round(4)
)

# Mostrar tabela com variâncias
print("\nVariância da utilização média dos PTDs por distrito:")
display(variancia.to_frame(name="Variância"))

### Interpretação dos Resultados

A distribuição da utilização média dos PTDs nos distritos de Lisboa, Porto, Aveiro e Setúbal foi analisada através de **boxplots**, permitindo observar a mediana, os quartis, a dispersão dos valores e a presença de possíveis **outliers** na variável `Util_Media`.

Visualmente, verifica-se que o distrito do **Porto apresenta maior dispersão**, evidenciada por uma caixa mais extensa e whiskers mais longos, indicando maior variabilidade nos níveis médios de utilização da rede.

Para complementar a análise gráfica, foi calculada a **variância da utilização média por distrito**, obtendo-se aproximadamente:

- Porto: **0.0065**
- Setúbal: **0.0051**
- Lisboa: **0.0026**
- Aveiro: **0.0018**

Estes resultados confirmam que o **distrito do Porto apresenta a maior variabilidade na utilização média dos PTDs**, enquanto **Aveiro apresenta uma distribuição mais homogénea** entre os concelhos analisados.

## Prevalência de valores omissos e identificação de outliers

Nesta etapa pretende-se analisar a qualidade dos dados disponíveis relativamente à utilização dos PTDs.

Algumas variáveis apresentam valores não determinados ou censurados, nomeadamente representados por **"N/D"** (não disponível) ou **"<20"** (valor inferior ao limite de divulgação). Estes valores não correspondem a observações numéricas válidas e devem ser tratados antes de qualquer análise estatística.

Assim, procede-se à identificação e quantificação destes valores no dataset. Posteriormente, os valores válidos são utilizados para analisar a distribuição da ocupação da rede e identificar a presença de **outliers**, recorrendo a representações gráficas adequadas.

In [ ]:
colunas_analise = [
    "Potência Geração [kW]",
    "Número de Clientes",
    "Número de Clientes Produtores"
]

resultados = []

for col in colunas_analise:

    total = len(ptd_data[col])
    nd = (ptd_data[col].astype(str) == "N/D").sum()
    lt20 = (ptd_data[col].astype(str) == "<20").sum()

    resultados.append({
        "Variável": col,
        "Total Registos": total,
        "N/D": nd,
        "% N/D": nd/total,
        "<20": lt20,
        "% <20": lt20/total
    })

df_resultados = pd.DataFrame(resultados)

display(
    df_resultados.style.format({
        "% N/D": "{:.2%}",
        "% <20": "{:.2%}"
    })
)

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(df_final["Util_Media"], vert=False)

plt.title("Identificação de outliers na utilização média da rede")
plt.xlabel("Utilização média dos PTDs")

plt.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

### Interpretação dos resultados

A análise do dataset original dos PTDs revelou a presença de valores indeterminados representados por `"N/D"` e `"<20"` em algumas variáveis.

Observa-se que a variável **Potência de Geração [kW]** apresenta uma proporção muito elevada de valores `"N/D"`, indicando que, na maioria dos PTDs, não existe produção associada ou essa informação não se encontra disponível no dataset. Por outro lado, a variável **Número de Clientes Produtores** apresenta predominantemente valores `"<20"`, o que significa que o número de produtores ligados ao PTD é inferior a 20 ou que o valor exato foi censurado por motivos de confidencialidade estatística.

Relativamente à ocupação da rede, o boxplot da variável **Util_Media** permite analisar a distribuição da utilização média dos PTDs entre os concelhos. Verifica-se que a maioria dos valores se concentra aproximadamente entre **46% e 56%**, com uma mediana próxima de **51%**.

Não se observam outliers extremamente pronunciados na distribuição agregada por concelho, sugerindo que os níveis médios de utilização da rede elétrica apresentam uma **variabilidade moderada e relativamente homogénea** entre os concelhos analisados.

## Estatísticas descritivas do nível de utilização médio

Nesta secção apresentam-se estatísticas descritivas da variável **Util_Media**, correspondente ao **nível médio de utilização dos PTDs**, para os concelhos de **Coimbra, Évora, Braga e Faro**.

As medidas calculadas incluem:

- média
- quartis
- desvio padrão
- assimetria
- curtose

Estas estatísticas permitem caracterizar a **distribuição da utilização média da rede elétrica** nos concelhos selecionados, identificando diferenças na variabilidade, forma da distribuição e possíveis assimetrias nos níveis de ocupação da infraestrutura elétrica.

In [ ]:
concelhos = ["Coimbra", "Évora", "Braga", "Faro"]

df_stats = ptd_data[ptd_data["Concelho"].isin(concelhos)]

tabela_434 = (
    df_stats.groupby("Concelho")["Utilização Decimal"]
    .agg(
        media="mean",
        Q1=lambda x: x.quantile(0.25),
        mediana="median",
        Q3=lambda x: x.quantile(0.75),
        desvio_padrao="std",
        assimetria="skew",
        curtose=lambda x: x.kurt()
    )
    .round(4)
)

display(tabela_434)

### Interpretação dos resultados

A tabela apresenta estatísticas descritivas do nível médio de utilização da rede elétrica nos concelhos de **Braga, Coimbra, Faro e Évora**.

Observa-se que os valores médios de utilização são semelhantes em **Braga (0.5423), Coimbra (0.5406) e Faro (0.5549)**, situando-se próximos de **54% de utilização média da capacidade dos PTDs**. Em contraste, **Évora apresenta uma média inferior (0.4546)**, indicando uma menor ocupação média da infraestrutura elétrica.

Os quartis mostram que a maior parte dos valores de utilização se concentra entre aproximadamente **0.39 e 0.79**, evidenciando uma variabilidade considerável entre os PTDs dentro de cada concelho.

O **desvio padrão**, situado entre cerca de **0.21 e 0.24**, confirma a existência de dispersão moderada nos níveis de utilização da rede.

Os valores positivos de **assimetria** indicam uma distribuição ligeiramente enviesada à direita, sugerindo a presença de alguns PTDs com níveis de utilização mais elevados.

Por fim, os valores negativos de **curtose** indicam distribuições relativamente mais achatadas do que a distribuição normal, refletindo uma dispersão mais uniforme dos níveis de utilização observados.

# 4.4 Inferência Estatística

* Considere a base de dados consolidada e selecione aleatoriamente uma amostra de 50 concelhos.
Use esta amostra para testar se o nível médio de ocupação da rede é inferior a um patamar de
referência (ex: 60\%), verificando previamente a normalidade dos dados.

* Selecione aleatoriamente duas amostras de 30 registos: uma de concelhos "Modernizados" (rácio de
LED acima da mediana) e outra de concelhos "Ineficientes". Use estas amostras para testar se o
estado médio de ocupação da rede difere significativamente entre os dois grupos.

* Considere três amostras aleatórias de 25 concelhos representativas de diferentes perfis de ocupação
da rede: Norte/Centro Litoral (Porto, Braga, Coimbra), Lisboa e Litoral Sul (Lisboa, Setúbal, Aveiro) e
Interior/Alentejo (Évora, Beja, Portalegre). Use estas amostras para testar a existência de diferenças
significativas nos níveis médios de carga da rede (ANOVA). Caso necessário, efetue uma análise post-
hoc adequada.

* Teste a existência de uma relação linear estatisticamente significativa entre a capacidade total de
transformação instalada e a carga de iluminação pública para a totalidade dos concelhos,
interpretando o coeficiente de correlação de Pearson.

Nesta secção, vamos realizar testes de hipóteses estatísticas com um nível de significância de 5% para tirar conclusões sobre a capacidade da rede e o impacto da iluminação pública.

In [ ]:
# Configurar diretório base do projeto
notebook_dir = Path.cwd()
if notebook_dir.name == "src": os.chdir(notebook_dir.parent)
df = pd.read_csv('data/dataset_final.csv')

### 4.4.1. Teste de Ocupação Média da Rede (Amostra Única)
**Objetivo:** Selecionar aleatoriamente uma amostra de 50 concelhos e testar se o nível médio de ocupação da rede (`Util_Media`) é inferior a 60%.

**Metodologia:** 1. Teste de Normalidade de Shapiro-Wilk para verificar as condições de aplicabilidade.
2. Teste T de uma amostra (One-Sample T-test) unilateral à esquerda.

In [ ]:
# 1. Select a random sample of 50 municipalities
amostra_50 = df['Util_Media'].dropna().sample(n=50, random_state=42)

# 2. Shapiro-Wilk Test for Normality
# H0: Data follows a normal distribution
stat_shapiro, p_shapiro = stats.shapiro(amostra_50)
print(f"Shapiro-Wilk p-value: {p_shapiro:.4f}")

# 3. One-Sample T-test
# H0: Mean >= 0.60 | H1: Mean < 0.60
stat_t, p_t = stats.ttest_1samp(amostra_50, popmean=0.60, alternative='less')
print(f"T-test p-value: {p_t:.4f}")

**Análise e Conclusão:**
* **Normalidade:** O teste de Shapiro-Wilk obteve um p-value de `0.0523`. Como este valor é ligeiramente maior que 0.05, não rejeitamos a hipótese nula, assumindo que os dados seguem uma distribuição normal.

* **Conclusão do Teste:** O teste T obteve um p-value de `0.0000`. A 5% de significância, rejeitamos a hipótese nula. Conclui-se que a ocupação média é estatisticamente inferior a 60%. Isto indica que a rede tem, em média, folga disponível antes de qualquer substituição de lâmpadas.

### 4.4.2. Comparação de Concelhos Modernizados vs. Ineficientes
**Objetivo:** Testar se o estado médio de ocupação da rede difere significativamente entre concelhos "Modernizados" e "Ineficientes" utilizando duas amostras de 30 registos.

**Metodologia:**
Divisão dos dados com base na mediana do rácio de LED e aplicação de um Teste T para duas amostras independentes.


In [ ]:
# Create LED ratio variable
df['Rate_LED'] = 1 - df['Rate_Ineficiencia']
mediana_led = df['Rate_LED'].median()

# Split the data
modernizados = df[df['Rate_LED'] > mediana_led]['Util_Media'].dropna()
ineficientes = df[df['Rate_LED'] <= mediana_led]['Util_Media'].dropna()

# Random samples of 30
amostra_mod = modernizados.sample(n=30, random_state=42)
amostra_inef = ineficientes.sample(n=30, random_state=42)

# Independent T-test (assuming equal variances for simplicity, but Levene's test can check this)
stat_ind, p_ind = stats.ttest_ind(amostra_mod, amostra_inef, equal_var=False)
print(f"Independent T-test p-value: {p_ind:.4f}")

**Análise e Conclusão:**
O Teste T para amostras independentes obteve um p-value de `0.3071`. Como este valor é maior que o nível de significância de 0.05, não rejeitamos a hipótese nula. Conclui-se que não existe uma diferença estatisticamente significativa no nível médio de ocupação da rede entre os concelhos com maior adoção de tecnologia LED e os concelhos mais ineficientes.

### 4.4.3. Análise de Variância (ANOVA) por Perfil Geográfico
**Objetivo:** Testar a existência de diferenças significativas nos níveis médios de carga da rede entre 3 perfis geográficos (Norte/Centro Litoral, Lisboa/Litoral Sul, Interior/Alentejo) usando amostras de 25 concelhos.

**Metodologia:** ANOVA a um fator (One-way ANOVA) seguida de análise post-hoc de Tukey caso se verifiquem diferenças.

In [ ]:
# Define groups based on Distritos
norte_centro = ['Porto', 'Braga', 'Coimbra']
lisboa_sul = ['Lisboa', 'Setúbal', 'Aveiro']
interior = ['Évora', 'Beja', 'Portalegre']

# Filter and sample 25
g1 = df[df['Distrito'].isin(norte_centro)]['Util_Media'].dropna().sample(n=25, random_state=42, replace=True) # Used replace=True just in case there aren't 25 available
g2 = df[df['Distrito'].isin(lisboa_sul)]['Util_Media'].dropna().sample(n=25, random_state=42, replace=True)
g3 = df[df['Distrito'].isin(interior)]['Util_Media'].dropna().sample(n=25, random_state=42, replace=True)

# Run ANOVA
stat_anova, p_anova = stats.f_oneway(g1, g2, g3)
print(f"ANOVA p-value: {p_anova:.4f}")

# Post-Hoc Analysis (Only if ANOVA p-value < 0.05)
if p_anova < 0.05:
    # Prepare data for Tukey
    tukey_data = np.concatenate([g1, g2, g3])
    tukey_labels = ['Norte/Centro']*25 + ['Lisboa/Sul']*25 + ['Interior']*25
    tukey_results = pairwise_tukeyhsd(tukey_data, tukey_labels, alpha=0.05)
    print("\nResultados Post-Hoc de Tukey:")
    print(tukey_results)
else:
    print("\nComo o p-value da ANOVA é > 0.05, não há diferenças significativas, logo não é necessário teste post-hoc.")

**Análise e Conclusão:**
A ANOVA resultou num p-value de `0.0000`. Assim, concluímos que existem diferenças significativas entre os níveis médios de carga das diferentes regiões.
A análise post-hoc de Tukey revelou que as diferenças significativas residem entre todos os pares de regiões testados (todas as comparações indicam reject=True). Ou seja, o perfil do Interior difere significativamente tanto de Lisboa/Sul como do Norte/Centro, e Lisboa/Sul também difere significativamente do Norte/Centro.

### 4.4.4. Correlação Linear: Capacidade vs. Iluminação Pública
**Objetivo:** Testar se existe uma relação linear significativa entre a capacidade total de transformação instalada e a carga de iluminação pública para todos os concelhos.

In [ ]:
# Drop missing values for these two specific columns
corr_df = df[['Cap_PTD', 'P_IP_TOTAL']].dropna()

# Calculate Pearson Correlation
r, p_corr = stats.pearsonr(corr_df['Cap_PTD'], corr_df['P_IP_TOTAL'])
print(f"Coeficiente de Correlação de Pearson (r): {r:.4f}")
print(f"P-value: {p_corr:.4f}")

**Análise e Conclusão:**
A correlação de Pearson testada indicou um coeficiente r = `0.9111`, com um p-value de `0.0000`. Como o p-value é menor que o nível de significância de 5%, a relação é estatisticamente significativa. O valor de r sugere que existe uma forte e positiva correlação linear. Isto significa que concelhos com maior carga de iluminação pública tendem a ter uma capacidade total instalada significativamente maior nos seus transformadores.

# Correlação e Regressão - Modelo Preditivo

Considere os dados relativos aos níveis médios de carga dos PTDs nos distritos de Aveiro, Porto, Lisboa e Braga e construa uma tabela de correlação entre as principais métricas de infraestrutura (Capacidade PTD e Potência IP) para estes distritos.


In [ ]:
# Carregar dados
csv_data = pd.read_csv('data/dataset_final.csv')

# Filtrar para os distritos especificados
distritos = ['Aveiro', 'Porto', 'Lisboa', 'Braga']
data_filtrado = csv_data[csv_data['Distrito'].isin(distritos)].copy()

# Configurar pandas para mostrar todos os resultados
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Dados filtrados para os distritos:')
print(data_filtrado[['Distrito', 'Concelho', 'Cap_PTD', 'P_IP_TOTAL', 'Util_Media']])

print(f'\nTotal de registos filtrados: {len(data_filtrado)}')

# Tabela de correlação
correlacao_infra = data_filtrado[['Cap_PTD', 'P_IP_TOTAL']].corr(method='pearson')
print('\nTabela de correlação (Cap_PTD vs P_IP_TOTAL):')
print(correlacao_infra)

# Correlação por distrito (complemento útil para comparar os 4 distritos)
correlacao_por_distrito = (
    data_filtrado
    .groupby('Distrito')
    .apply(lambda g: g[['Cap_PTD', 'P_IP_TOTAL']].corr().iloc[0, 1])
    .reset_index(name='Corr_Pearson_Cap_PTD_P_IP_TOTAL')
)

print('\nCorrelação de Pearson por distrito (Cap_PTD vs P_IP_TOTAL):')
print(correlacao_por_distrito)

---

Selecione os dados relativos a Portugal Continental. Considere as seguintes variáveis explicativas:

- X1 - Potência Instalada Total de Iluminação Pública (`P_IP_Total`)
- X2 - Capacidade Nominal de Transformação (`Cap_PTD`)
- X3 - Ineficiência (Rate_Ineficiencia) e a variável dependente:
- Y - Estado de Ocupação Médio da Rede (`Util_Media`).

---

In [ ]:
# Selecionar dados de Portugal Continental
portugal_continental = data_filtrado.copy()

# Selecionar as variáveis explicativas e dependente
X1 = portugal_continental['P_IP_TOTAL']        # Potência Instalada Total de Iluminação Pública
X2 = portugal_continental['Cap_PTD']           # Capacidade Nominal de Transformação
X3 = portugal_continental['Rate_Ineficiencia']  # Ineficiência
Y = portugal_continental['Util_Media']         # Estado de Ocupação Médio da Rede

# Criar dataframe com as variáveis
dados_analise = pd.DataFrame({
    'X1_P_IP_TOTAL': X1,
    'X2_Cap_PTD': X2,
    'X3_Rate_Ineficiencia': X3,
    'Y_Util_Media': Y
})

print('Dados selecionados para análise:')
print(dados_analise)

1. Determine o modelo de regressão linear múltipla que explique a variação de Y em função de X1, X2 e X3.

In [ ]:
# Remover valores em falta
dados_limpos = dados_analise.dropna()

# Preparar as variáveis explicativas (X) e dependentes (Y)
X = dados_limpos[['X1_P_IP_TOTAL', 'X2_Cap_PTD', 'X3_Rate_Ineficiencia']]
Y_var = dados_limpos[['Y_Util_Media']]

# Adicionar constante para o termo independente
X_constante = sm.add_constant(X)

# Criar a ajustar o modelo
modelo_regressao = sm.OLS(Y_var, X_constante).fit()

# Mostrar resumo do modelo
print("MODELO DE REGRESSÃO LINEAR MÚLTIPLA")
print(modelo_regressao.summary())

In [ ]:
# Mostrar equação do modelo
print("="*70)

print("EQUAÇÃO DO MODELO")

const = modelo_regressao.params['const']
coef_x1 = modelo_regressao.params['X1_P_IP_TOTAL']
coef_x2 = modelo_regressao.params['X2_Cap_PTD']
coef_x3 = modelo_regressao.params['X3_Rate_Ineficiencia']

print(f"Y = {const:.6f} + {coef_x1:.6f} * X1 + {coef_x2:.6f} * X2 + {coef_x3:.6f} * X3")

# Mostrar Métricas Principais
print("="*70)

print("MÉTRICAS PRINCIPAIS")

print("="*70)
print(f"R-squared (R²): {modelo_regressao.rsquared:.6f}")
print(f"Adjusted R-squared: {modelo_regressao.rsquared_adj:.6f}")
print(f"F-statistic: {modelo_regressao.fvalue:.6f}")
print(f"Prob (F-statistic): {modelo_regressao.f_pvalue:.6e}")

print("="*70)

2. Verifique as condições sobre os resíduos (normalidade, independência e homocedasticidade).

In [ ]:
# Obter os resíduos
residuos = modelo_regressao.resid

print("Análise dos Resíduos")

# Normalidade
print("\n1. TESTE DE NORMALIDADE")

sm.qqplot(residuos, line = 's', markersize = 5, alpha = 0.5)

plt.title('Residuos de NORMALIDADE')
plt.show()

In [ ]:
# Teste de Shapiro
stat_shapiro, p_value_shapiro = shapiro(residuos)

print(f"Teste de Shapiro-Wilk:")
print(f"  Estatística: {stat_shapiro:.6f}")
print(f"  P-value: {p_value_shapiro:.6f}")

if p_value_shapiro > 0.05:
    print(f"Conclusão: Os resíduos parecem ser normalmente distribuídos (p > 0.05)")
else:
    print(f"Conclusão: Os resíduos não parecem ser normalmente distribuídos (p < 0.05)")

In [ ]:
# Homocedasticidade

plt.scatter(modelo_regressao.fittedvalues, residuos, alpha = 0.5)
plt.axhline(y = 0, color = 'r', linestyle = '--')

plt.title('Gráfico Homocidasticidade')

plt.xlabel('Valores Ajustados')
plt.ylabel('Residuos')

plt.show()

## HOMOCEDASTICIDADE

**Conclusão:**

Da análise do gráfico de resíduos com os valores ajustados resulta que a condição de homocedasticidade é verificada.

#### Observações Principais:

1. **Dispersão dos Resíduos:**
   - Os resíduos apresentam uma distribuição relativamente simétrica em torno da linha de referência (y=0)
   - Não existe um padrão claro de funil (aumento/diminuição sistemática da variância)

2. **Variância Residual:**
   - A variância dos resíduos mantém-se aproximadamente constante ao longo do intervalo de valores ajustados
   - Isto sugere que a presença de homocedasticidade não é claramente violada

3. **Possíveis Desvios:**
   - Alguns resíduos extremos podem indicar heterocedacidade ligeira nas extremidades
   - A amostra relativamente pequena limita a conclusão definitiva

#### Conclusão Final:

A condição de **homocedasticidade é razoavelmente satisfeita**, permitindo a utilização válida do modelo OLS.

---

In [ ]:
# Independência dos Resíduos
durbinWatson = durbin_watson(residuos)

print('Valor da estatística DW: ', durbinWatson)

## INDEPENDÊNCIA DOS RESÍDUOS (Teste de Durbin-Watson)

**Teste Utilizado:** Teste de Durbin-Watson (DW)

**Interpretação da Estatística DW:**

O teste de Durbin-Watson deteta autocorrelação nos resíduos. A estatística DW varia entre 0 e 4:

- **DW ≈ 2:** Ausência de autocorrelação (cenário ideal)
- **DW < 2:** Autocorrelação positiva (resíduos consecutivos tendem a ser semelhantes)
- **DW > 2:** Autocorrelação negativa (resíduos consecutivos tendem a ser opostos)
- **DW ≈ 0 ou DW ≈ 4:** Autocorrelação forte

---

**Resultado Obtido:** DW = 2.0156001981515197. A estatística confirma a ausência de autocorrelação nos resíduos. O valor está próximo do ideal indicando que os resíduos são independentes e não apresentam padrões de autocorrelação problemáticos. Significa que os erros não dependem uns dos outros, não existe correlação entre resíduos consecutivos, o modelo OLS é válido neste aspeto e os testes estatísticos e intervalos de confiança são confiáveis.

---

3. Verifique se existe multicolinearidade entre as variáveis recorrendo ao cálculo do VIF (Variance Inflation Factor).

In [ ]:
print("ANÁLISE DE MULTICOLINEARIDADE - VIF (Variance Inflation Factor)")
print("="*70)

# Calcular VIF para variável explicativa
X_para_vif = X.copy()

# Criar dataframe com os VIFs
vif_data = pd.DataFrame()
vif_data["Variável"] = X_para_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_para_vif.values, i) for i in range(X_para_vif.shape[1])]

print("\nValores de VIF:")
print(vif_data)

---

# Conclusão

- Os valores das variáveis X1_P_IP_TOTAL e X2_Cap_PTD apresentam valores significativamente superiores a 5, o que indica a existência de multicolinearidade entre elas.

- A presença desta multicolinearidade implica que o cálculo dos coeficientes de regressão será numericamente instável. Isto significa que pequenas variações nos dados podem provocar grandes alterações dos valores dos coefientes estimados, retirando fiabilidade à interpretação individual de cada variável.

- A variável X3_Rate_Ineficiencia é a única que se encontra dentro do intervalo aceitável (VIF < 5), não contribuindo para o problema de colinearidade no modelo.

---

4. Comente o modelo obtido tendo em conta o coeficiente de determinação ajustado e a significância estatística dos preditores.

In [ ]:
# Extrair informações principais do modelo
r_squared = modelo_regressao.rsquared
r_squared_adj = modelo_regressao.rsquared_adj
f_stat = modelo_regressao.fvalue
p_value_f = modelo_regressao.f_pvalue
pvalues = modelo_regressao.pvalues

print("\n1. COEFICIENTE DE DETERMINAÇÃO AJUSTADO (R² Ajustado)")
print("="*70)

print(f"R² (não ajustado): {r_squared:.6f}")
print(f"R² Ajustado: {r_squared_adj:.6f}")
print(f"Diferença: {r_squared - r_squared_adj:.6f}")

print("="*70)

---

Interpretação:

- O modelo explica 23.79% da variabilidade total na variável dependente (Y = Util_Media).
- Isto significa que ~76.21% da variabilidade em Y é explicada por fatores não incluídos no modelo.
- Um R² ajustado de 0.238 é considerado baixo/moderado, indicando que o modelo tem capacidade preditiva limitada.

---

In [ ]:
print("\n2. SIGNIFICÂNCIA ESTATÍSTICA DOS PREDITORES")
print("-"*70)

# Criar tabela com pvalues
pvalues_table = pd.DataFrame({
    'Preditor': ['Constante', 'X1_P_IP_TOTAL', 'X2_Cap_PTD', 'X3_Rate_Ineficiencia'],

    'Coeficiente': [const, coef_x1, coef_x2, coef_x3],

    'P-value': [pvalues['const'], pvalues['X1_P_IP_TOTAL'],
                pvalues['X2_Cap_PTD'], pvalues['X3_Rate_Ineficiencia']],

    'Significância': ['***' if pvalues['const'] < 0.001 else '**' if pvalues['const'] < 0.01 else '*' if pvalues['const'] < 0.05 else 'NS',
                      '***' if pvalues['X1_P_IP_TOTAL'] < 0.001 else '**' if pvalues['X1_P_IP_TOTAL'] < 0.01 else '*' if pvalues['X1_P_IP_TOTAL'] < 0.05 else 'NS',
                      '***' if pvalues['X2_Cap_PTD'] < 0.001 else '**' if pvalues['X2_Cap_PTD'] < 0.01 else '*' if pvalues['X2_Cap_PTD'] < 0.05 else 'NS',
                      '***' if pvalues['X3_Rate_Ineficiencia'] < 0.001 else '**' if pvalues['X3_Rate_Ineficiencia'] < 0.01 else '*' if pvalues['X3_Rate_Ineficiencia'] < 0.05 else 'NS']
})

print("\nTabela de P-values dos Preditores:")
print(pvalues_table)
print("\nLegenda: *** p<0.001 | ** p<0.01 | * p<0.05 | NS = Não Significativo")

print("\nAnálise individual dos preditores:")
print("-"*70)

# X1 - P_IP_TOTAL
print(f"\n• X1 (Potência Instalada Total - P_IP_TOTAL):")
print(f"  - Coeficiente: {coef_x1:.6e}")
print(f"  - P-value: {pvalues['X1_P_IP_TOTAL']:.4f}")

if pvalues['X1_P_IP_TOTAL'] < 0.05:
    print(f"  - Resultado: SIGNIFICATIVO ao nível 0.05 (p={pvalues['X1_P_IP_TOTAL']:.4f})")
    print(f"  - Interpretação: Um aumento de 1 unidade em X1 causa um aumento")
    print(f"    de {coef_x1:.6e} na utilização média da rede.")
else:
    print(f"  - Resultado: NÃO SIGNIFICATIVO ao nível 0.05 (p={pvalues['X1_P_IP_TOTAL']:.4f})")
    print(f"  - Interpretação: Não há evidência suficiente de que X1 afeta Y.")

# X2 - Cap_PTD

print(f"\n• X2 (Capacidade Nominal de Transformação - Cap_PTD):")
print(f"  - Coeficiente: {coef_x2:.6e}")
print(f"  - P-value: {pvalues['X2_Cap_PTD']:.4f}")

if pvalues['X2_Cap_PTD'] < 0.05:
    print(f"  - Resultado: SIGNIFICATIVO ao nível 0.05 (p={pvalues['X2_Cap_PTD']:.4f})")
    print(f"  - Interpretação: Um aumento de 1 unidade em X2 causa uma diminuição de {abs(coef_x2):.6e} na utilização média da rede.")
else:
    print(f"  - Resultado: NÃO SIGNIFICATIVO ao nível 0.05 (p={pvalues['X2_Cap_PTD']:.4f})")
    print(f"  - Interpretação: Não há evidência suficiente de que X2 afeta Y.")

# X3 - Rate_Ineficiencia

print(f"\n• X3 (Ineficiência - Rate_Ineficiencia):")
print(f"  - Coeficiente: {coef_x3:.6f}")
print(f"  - P-value: {pvalues['X3_Rate_Ineficiencia']:.4f}")

if pvalues['X3_Rate_Ineficiencia'] < 0.05:
    print(f"  - Resultado: SIGNIFICATIVO ao nível 0.05 (p={pvalues['X3_Rate_Ineficiencia']:.4f})")
    print(f"  - Interpretação: Um aumento de 1 unidade em X3 causa um aumento de {coef_x3:.6f} na utilização média da rede.")
else:
    print(f"  - Resultado: NÃO SIGNIFICATIVO ao nível 0.05 (p={pvalues['X3_Rate_Ineficiencia']:.4f})")
    print(f"  - Interpretação: Não há evidência suficiente de que X3 afeta Y.")

---

**Resultado:** O modelo é GLOBALMENTE SIGNIFICATIVO (p<0.05)")

**Interpretação:** As variáveis explicativas, em conjunto, têm uma relação estatisticamente significativa com a variável dependente.

---

---

## 3. SIGNIFICÂNCIA GLOBAL DO MODELO

**Estatísticas Gerais do Modelo:**

- **F-statistic:** 7.866744
- **P-value (F-statistic):** 1.544066e-04 (≈ 0.0001544)

**Resultado:** O modelo é significativo (p < 0.05)

**Conclusão e Interpretação:**

Com um p-value de 0.0001544 (muito inferior ao nível de significância de 0.05), o modelo apresenta significância estatística global.

As variáveis explicativas (X1 - Potência IP, X2 - Capacidade PTD, X3 - Taxa de Ineficiência), em conjunto, têm uma relação **estatisticamente significativa** com a variável dependente (Y - Utilização Média da Rede).

O valor de F-statistic = 7.866744 indica que a razão entre a variância explicada pelo modelo e a variância dos resíduos é significativamente superior a 1.

**Implicações Práticas:**

- O modelo todo é estatisticamente válido e rejeita a hipótese nula (H₀: todos os coeficientes = 0)
- Existe evidência estatística significativa de que pelo menos uma das variáveis explicativas influencia a utilização média da rede
- Apesar da significância global, a capacidade preditiva individual das variáveis vária (conforme análise anterior)
- A presença de multicolinearidade (entre X1 e X2) pode comprometer a interpretação individual dos coeficientes

---


5. Estime o nível de ocupação esperado (Y) para os concelhos com os códigos: 101, 102, 103, 104, 105, 106, 107, 108 e 109 (Distrito de Aveiro) e compare as previsões com os valores reais presentes no dataset.

In [ ]:
print("PREVISÕES PARA OS CONCELHOS DO DISTRITO DE AVEIRO (Códigos 101-109)")

# Filtrar os dados originais para os concelhos específicos
codigos_aveiro = [101, 102, 103, 104, 105, 106, 107, 108, 109]
dados_aveiro_predic = csv_data[csv_data['CodDistritoConcelho'].isin(codigos_aveiro)].copy()

print(f"\nTotal de concelhos a prever: {len(dados_aveiro_predic)}")

# Selecionar apenas as variáveis necessárias para previsão
X_predicao = dados_aveiro_predic[['P_IP_TOTAL', 'Cap_PTD', 'Rate_Ineficiencia']].copy()
X_predicao_constante = sm.add_constant(X_predicao)

# Fazer previsões
previsoes = modelo_regressao.predict(X_predicao_constante)

# Valores reais
valores_reais = dados_aveiro_predic['Util_Media'].values

# Criar dataframe de comparação
comparacao = pd.DataFrame({
    'CodConcelho': dados_aveiro_predic['CodDistritoConcelho'].values,
    'Concelho': dados_aveiro_predic['Concelho'].values,
    'X1_P_IP_TOTAL': X_predicao['P_IP_TOTAL'].values,
    'X2_Cap_PTD': X_predicao['Cap_PTD'].values,
    'X3_Rate_Ineficiencia': X_predicao['Rate_Ineficiencia'].values,
    'Y_Real': valores_reais,
    'Y_Previsto': previsoes.values,
    'Erro': valores_reais - previsoes.values,
    'Erro_Absoluto': np.abs(valores_reais - previsoes.values),
    'Erro_Percentual': (np.abs(valores_reais - previsoes.values) / valores_reais * 100)
})

# Ordenar por código de concelho
comparacao = comparacao.sort_values('CodConcelho')

print("TABELA COMPARATIVA - VALORES REAIS vs PREVISTOS")
print(comparacao)

# Calcular métricas de erro
print("MÉTRICAS DE ERRO GLOBAL")

mae = np.mean(comparacao['Erro_Absoluto'])
rmse = np.sqrt(np.mean((comparacao['Erro'])**2))
mape = np.mean(comparacao['Erro_Percentual'])

print(f"\nMAE (Mean Absolute Error): {mae:.6f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.6f}")
print(f"MAPE (Mean Absolute Percentage Error): {mape:.2f}%")

print(f"\nErro máximo: {comparacao['Erro_Absoluto'].max():.6f}")
print(f"Erro mínimo: {comparacao['Erro_Absoluto'].min():.6f}")
print(f"Erro médio: {comparacao['Erro'].mean():.6f}")

## VISUALIZAÇÃO GRÁFICA DAS PREVISÕES

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

#======================================================================================================================================

# Gráfico 1: Valores Reais vs Previstos
axes[0, 0].scatter(comparacao['Concelho'], comparacao['Y_Real'], label = 'Real', marker = 'o', s = 100, alpha = 0.7)
axes[0, 0].scatter(comparacao['Concelho'], comparacao['Y_Previsto'], label = 'Previsto', marker= 's', s = 100, alpha = 0.7)

axes[0, 0].set_xlabel('Concelho')
axes[0, 0].set_ylabel('Utilização Média (Y)')

axes[0, 0].set_title('Comparação: Valores Reais vs Previstos')

axes[0, 0].legend()

axes[0, 0].tick_params(axis = 'x', rotation = 45)

axes[0, 0].grid(True, alpha = 0.3)

#======================================================================================================================================

# Gráfico 2: Erros
cores_erro = ['red' if e < 0 else 'green' for e in comparacao['Erro']]
axes[0, 1].bar(comparacao['Concelho'], comparacao['Erro'], color=cores_erro, alpha=0.7)
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].set_xlabel('Concelho')
axes[0, 1].set_ylabel('Erro (Real - Previsto)')
axes[0, 1].set_title('Erros de Previsão por Concelho')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3, axis='y')

#======================================================================================================================================

# Gráfico 3: Erro Absoluto
axes[1, 0].bar(comparacao['Concelho'], comparacao['Erro_Absoluto'], color='orange', alpha=0.7)
axes[1, 0].set_xlabel('Concelho')
axes[1, 0].set_ylabel('Erro Absoluto')
axes[1, 0].set_title('Erro Absoluto por Concelho')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3, axis='y')

#======================================================================================================================================

# Gráfico 4: Dispersão (Real vs Previsto)
axes[1, 1].scatter(comparacao['Y_Real'], comparacao['Y_Previsto'], s=100, alpha=0.7)
# Adicionar linha de perfeição
y_min = min(comparacao['Y_Real'].min(), comparacao['Y_Previsto'].min())
y_max = max(comparacao['Y_Real'].max(), comparacao['Y_Previsto'].max())
axes[1, 1].plot([y_min, y_max], [y_min, y_max], 'r--', linewidth=2, label='Perfeição')
axes[1, 1].set_xlabel('Valores Reais')
axes[1, 1].set_ylabel('Valores Previstos')
axes[1, 1].set_title('Scatter Plot: Real vs Previsto')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

#======================================================================================================================================

plt.tight_layout()
plt.show()

In [ ]:
# Análise por concelho
print("ANÁLISE DETALHADA POR CONCELHO")

for idx, row in comparacao.iterrows():

    print(f"\n{row['CodConcelho']} - {row['Concelho']}:")
    print(f"  Valor Real: {row['Y_Real']:.6f}")
    print(f"  Valor Previsto: {row['Y_Previsto']:.6f}")
    print(f"  Erro: {row['Erro']:.6f}")
    print(f"  Erro Percentual: {row['Erro_Percentual']:.2f}%")

    if abs(row['Erro']) < 0.05:
        print(f"  Avaliação: Excelente previsão")
    elif abs(row['Erro']) < 0.1:
        print(f"  Avaliação: Boa previsão")
    else:
        print(f"  Avaliação: Previsão fraca")

6. Com base no coeficiente $\beta_{3}$ obtido no modelo, quantifique a redução esperada no nível de ocupação da rede caso o rácio de ineficiência tecnológica seja reduzido em 20% através da implementação de tecnologia LED.


### DISCLAIMER CRÍTICO: LIMITAÇÕES ESTATÍSTICAS

#### Não-Significância do Coeficiente β₃
O coeficiente β₃ **NÃO É ESTATISTICAMENTE SIGNIFICATIVO**
- **P-value:** 0.7959 (>> 0.05)
- **Conclusão:** A relação entre X3 (ineficiência) e Y (utilização) **não tem validade estatística**

**As análises a seguir são TEÓRICAS e baseadas num coeficiente não-significativo. Devem ser interpretadas com EXTREMA CAUTELA.**

---

#### Capacidade Preditiva Limitada
- **R² ajustado:** 0.238
- **Variância explicada:** apenas 23.8%
- **Variância não explicada:** 76.2%

---


In [ ]:
print("ANÁLISE DE IMPACTO - REDUÇÃO DE INEFICIÊNCIA COM TECNOLOGIA LED")
print("="*70)

# Extrair o coeficiente β₃
beta_3 = coef_x3

print(f"\nCoeficiente β₃ (Rate_Ineficiencia): {beta_3:.6f}")
print(f"P-value de X3: {pvalues['X3_Rate_Ineficiencia']:.4f} (NÃO SIGNIFICATIVO)")

# Calcular o valor médio de X3
X3_medio = dados_limpos['X3_Rate_Ineficiencia'].mean()
print(f"\nValor médio de X3 (Rate_Ineficiencia) no dataset: {X3_medio:.6f}")

In [ ]:
# Redução de 20% em X3
percentual_reducao = 0.20
X3_novo = X3_medio * (1 - percentual_reducao)
delta_X3 = X3_novo - X3_medio

print("REDUÇÃO DE 20% NA INEFICIÊNCIA TECNOLÓGICA (Implementação LED)")

print(f"\nX3 inicial (médio): {X3_medio:.6f}")
print(f"Redução: 20%")
print(f"X3 novo: {X3_novo:.6f}")
print(f"Variação ΔX3: {delta_X3:.6f}")

In [ ]:
# Calcular a mudança esperada em Y
delta_Y = beta_3 * delta_X3

print("CÁLCULO DO IMPACTO NA UTILIZAÇÃO MÉDIA (Y)")

print(f"\nFórmula: ΔY = β₃ × ΔX3")
print(f"ΔY = {beta_3:.6f} × {delta_X3:.6f}")
print(f"ΔY = {delta_Y:.6f}")

In [ ]:
# Interpretação

percentual_y = (abs(delta_Y) / dados_limpos['Y_Util_Media'].mean()) * 100

if delta_Y < 0:
    print(f"\nRESULTADO: Redução esperada no nível de ocupação de {abs(delta_Y):.6f}")
    print(f"  (equivalente a uma redução de ~{percentual_y:.2f}% relativamente à utilização média atual)")

elif delta_Y > 0:
    print(f"\nRESULTADO: Aumento esperado no nível de ocupação de {delta_Y:.6f}")
    print(f"  (isto sugere que a redução de ineficiência aumentaria a ocupação)")

else:
    print(f"\n RESULTADO: Sem impacto esperado (ΔY ≈ 0)")


In [ ]:
# Visualização do impacto
print("VISUALIZAÇÃO DO IMPACTO")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

#======================================================================================================================================

# Gráfico 1: Comparação Y atual vs Y após LED
axes[0].bar(comparacao['Concelho'], comparacao['Y_Real'], label = 'Y Atual', alpha = 0.7, color='blue')
axes[0].bar(comparacao['Concelho'], comparacao['Y_Novo_Estimado'], label = 'Y Estimado', alpha = 0.7, color='green')

axes[0].set_xlabel('Concelho')
axes[0].set_ylabel('Utilização Média (Y)')

axes[0].set_title('Comparação: Utilização Atual vs Após Implementação LED')

axes[0].legend()

axes[0].tick_params(axis = 'x', rotation = 45)

axes[0].grid(True, alpha = 0.3)

#======================================================================================================================================

# Gráfico 2: Mudança em Y (Delta_Y)
cores = ['green' if x < 0 else 'red' for x in comparacao['Delta_Y']]
axes[1].bar(comparacao['Concelho'], comparacao['Delta_Y'], color=cores, alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Concelho')
axes[1].set_ylabel('Mudança em Y (ΔY)')
axes[1].set_title('Redução Esperada na Utilização com Implementação LED')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

#======================================================================================================================================

plt.tight_layout()
plt.show()

## CONCLUSÕES E RECOMENDAÇÕES - ANÁLISE DE IMPACTO LED

### 1. COEFICIENTE β₃ INTERPRETAÇÃO

- O coeficiente β₃ = 0.007800 indica que um aumento unitário na taxa de ineficiência tecnológica está associado a um aumento de 0.007800 na utilização média da rede.
- Isto significa que quanto maior a ineficiência, maior é o nível de ocupação, o que é contraintuitivo do ponto de vista técnico.

---

### 2. IMPACTO DA IMPLEMENTAÇÃO DE TECNOLOGIA LED (redução de 20%)

- Uma redução de 20% na ineficiência (X3) levaria a uma redução esperada de aproximadamente 0.001248 no nível de ocupação médio (~0.22% em média).
- Por concelho, as reduções variam dependendo dos valores atuais de ineficiência em cada localidade.
- Valor médio de Y atual: aproximadamente 0.519 (Util_Media)
- Valor médio de Y após LED estimado: ~0.518 (redução de ~0.001)

---

### 3. IMPLICAÇÕES PRÁTICAS

- A implementação de tecnologia LED poderia contribuir para uma redução do nível de ocupação da rede em aproximadamente 0.22%.
- Isto sugere uma melhoria na eficiência do sistema de iluminação pública, resultando em menor carga na rede elétrica.
- Embora a redução percentual pareça pequena, em redes de grande escala esta poupança pode representar economias significativas.

---

### 4. LIMITAÇÕES DO MODELO

- O R² ajustado baixo (0.2382) sugere que existem outros fatores não considerados que influenciam a utilização da rede.
- A forte multicolinearidade entre X1 e X2 (com VIF > 1000) compromete a confiabilidade dos coeficientes, incluindo β₃.
- As previsões devem ser interpretadas com cautela e como indicativas, não como valores absolutos.
- **O coeficiente β₃ é estatisticamente NÃO significativo (p = 0.7959), o que questiona fundamentalmente a validade desta relação.**

---

7. Utilize o modelo para identificar os concelhos onde a libertação de potência é estatisticamente mais viável para a instalação de carregadores de veículos elétricos de 22 kW , justificando com base nos intervalos de confiança das previsões.

# ANÁLISE DE VIABILIDADE - INSTALAÇÃO DE CARREGADORES DE VE 22 kW

## NOTA IMPORTANTE

O coeficiente β₃ (ineficiência) é **NÃO SIGNIFICATIVO** estatisticamente (p=0.7959).

Portanto, o **Score de Viabilidade** NÃO é baseado em Delta_Y teórico, mas sim em:

1. **Qualidade da previsão (30%)** - medida pela amplitude do IC
2. **Potência disponível (40%)** - potencial real (folga + ganho LED)
3. **Ganho potencial com LED (30%)** - redução de ineficiência mensurável

---


In [ ]:

# Obter os intervalos de confiança das previsões (95%)
previsoes_com_intervalo = modelo_regressao.get_prediction(X_predicao_constante)
intervalo_confianca = previsoes_com_intervalo.conf_int(alpha=0.05)
previsoes_erro_padrao = previsoes_com_intervalo.se_mean

# Adicionar intervalos de confiança ao dataframe de comparação
# intervalo_confianca é um numpy array, usar indexação direta
comparacao['Y_Previsto_IC_Inferior'] = intervalo_confianca[:, 0]
comparacao['Y_Previsto_IC_Superior'] = intervalo_confianca[:, 1]
comparacao['Margem_Erro'] = previsoes_erro_padrao
comparacao['Amplitude_IC'] = comparacao['Y_Previsto_IC_Superior'] - comparacao['Y_Previsto_IC_Inferior']

# Adicionar dados sobre potência disponível
comparacao['PFolga'] = dados_aveiro_predic['PFolga'].values  # Potência de folga
comparacao['Ganho_LED'] = dados_aveiro_predic['Ganho_LED'].values  # Potência a ganhar com LED

print("TABELA 1: PREVISÕES COM INTERVALOS DE CONFIANÇA 95%")

tabela_ic = comparacao[['CodConcelho', 'Concelho', 'Y_Real', 'Y_Previsto', 'Y_Previsto_IC_Inferior', 'Y_Previsto_IC_Superior', 'Amplitude_IC']].copy()

print(tabela_ic)

print("ANÁLISE DE VIABILIDADE PARA CARREGADORES DE 22 kW")

# Parâmetro de potência por carregador
potencia_carregador = 22  # kW

# Calcular potência disponível potencial após LED
comparacao['Potencia_Disponivel_Potencial'] = comparacao['PFolga'] + comparacao['Ganho_LED']

# Calcular número de carregadores que poderiam ser instalados
# Baseado na potência disponível potencial (folga + ganho LED)

# Margem de segurança: amplitude completa do intervalo de confiança
# Representa a incerteza nas previsões (quanto maior a amplitude, maior a incerteza)
comparacao['Margem_Seguranca'] = comparacao['Amplitude_IC']

# Número de carregadores 22 kW que podem ser instalados
comparacao['N_Carregadores_Estimado'] = (comparacao['Potencia_Disponivel_Potencial'] / potencia_carregador).astype(int)

# Score de viabilidade (0-100)
# Baseado em: margem de confiança (30%), potência disponível (40%) e redução esperada (30%)
# Normalização de cada componente para escala 0-100

# Componente 1: Qualidade da previsão (inversão da amplitude do IC)
# IC menor = maior confiança, maior score
amplitude_norm = comparacao['Amplitude_IC'].max() - comparacao['Amplitude_IC']
score_ic = (amplitude_norm / amplitude_norm.max() * 100) * 0.30  # 30% peso

# Componente 2: Potência disponível
# Mais potência = maior score
score_potencia = (comparacao['Potencia_Disponivel_Potencial'] / comparacao['Potencia_Disponivel_Potencial'].max() * 100) * 0.40  # 40% peso

# Componente 3: Potencial de melhoria (usar Ganho_LED em vez de Delta_Y não-significativo)
# Maior ganho LED = maior potencial de folga = maior score
score_ganho = (comparacao['Ganho_LED'] / comparacao['Ganho_LED'].max() * 100) * 0.30  # 30% peso

comparacao['Score_Viabilidade'] = score_ic + score_potencia + score_ganho

In [ ]:
print("TABELA 2: ANÁLISE DE POTÊNCIA E VIABILIDADE")

tabela_viabilidade = comparacao[['CodConcelho', 'Concelho', 'PFolga', 'Ganho_LED',
                                  'Potencia_Disponivel_Potencial', 'N_Carregadores_Estimado',
                                  'Amplitude_IC', 'Score_Viabilidade']].copy()

# Ordenar por Score_Viabilidade descendente
tabela_viabilidade = tabela_viabilidade.sort_values('Score_Viabilidade', ascending=False)

print(tabela_viabilidade.to_string(index=False))

print("CLASSIFICAÇÃO DE VIABILIDADE POR CONCELHO")

# Classificar por viabilidade
for idx, row in tabela_viabilidade.iterrows():
    score = row['Score_Viabilidade']

    if score >= 75:
        classificacao = "MUITO VIÁVEL"
    elif score >= 60:
        classificacao = "VIÁVEL"
    elif score >= 45:
        classificacao = "MODERADAMENTE VIÁVEL"
    else:
        classificacao = "NÃO VIÁVEL"

    print(f"\n{row['CodConcelho']} - {row['Concelho']}:")
    print(f"  Score de Viabilidade: {score:.1f}/100 - {classificacao}")
    print(f"  Potência Disponível Potencial: {row['Potencia_Disponivel_Potencial']:.2f} kW")
    print(f"  Carregadores 22 kW Estimados: {row['N_Carregadores_Estimado']}")
    print(f"  Amplitude do IC: {row['Amplitude_IC']:.6f} (margem de confiança)")

In [ ]:
# Identificar TOP 3 concelhos mais viáveis
print("TOP 3 CONCELHOS MAIS VIÁVEIS PARA INSTALAÇÃO DE CARREGADORES DE VE 22 kW")

top_3 = tabela_viabilidade.head(3)

for rank, (idx, row) in enumerate(top_3.iterrows(), 1):
    print(f"\n #{rank} - {row['Concelho']} (Código: {row['CodConcelho']})")
    print(f"   Score: {row['Score_Viabilidade']:.1f}/100")
    print(f"   Potência Disponível: {row['Potencia_Disponivel_Potencial']:.2f} kW")
    print(f"   Carregadores Estimados: {row['N_Carregadores_Estimado']} × 22 kW")
    print(f"   Justificação Estatística:")

    # Obter dados do comparacao para justificação
    concelho_data = comparacao[comparacao['CodConcelho'] == row['CodConcelho']].iloc[0]

    print(f"     - Intervalo de Confiança 95%: [{concelho_data['Y_Previsto_IC_Inferior']:.6f}, {concelho_data['Y_Previsto_IC_Superior']:.6f}]")
    print(f"     - Margem de Segurança: {concelho_data['Margem_Seguranca']:.6f}")
    print(f"     - Erro Padrão: {concelho_data['Margem_Erro']:.6f}")
    print(f"     - Ganho Potencial com LED: {concelho_data['Ganho_LED']:.2f} kW")

In [ ]:
# Visualização gráfica
print("VISUALIZAÇÃO GRÁFICA - VIABILIDADE POR CONCELHO")

fig, axes = plt.subplots(2, 2, figsize = (16, 12))

# Gráfico 1: Score de Viabilidade
tabela_viabilidade_sorted = tabela_viabilidade.sort_values('Score_Viabilidade', ascending = True)
cores_score = ['green' if x >= 75 else 'yellow' if x >= 60 else 'orange' if x >= 45 else 'red'
               for x in tabela_viabilidade_sorted['Score_Viabilidade']]
axes[0, 0].barh(tabela_viabilidade_sorted['Concelho'], tabela_viabilidade_sorted['Score_Viabilidade'], color = cores_score, alpha = 0.7)
axes[0, 0].axvline(x = 75, color = 'green', linestyle = '--', linewidth = 2, label = 'Muito Viável (75)')
axes[0, 0].axvline(x = 60, color = 'yellow', linestyle = '--', linewidth = 2, label = 'Viável (60)')
axes[0, 0].axvline(x = 45, color = 'orange', linestyle = '--', linewidth = 2, label = 'Moderadamente Viável (45)')

axes[0, 0].set_xlabel('Score de Viabilidade')

axes[0, 0].set_title('Score de Viabilidade para Instalação de Carregadores VE 22kW')

axes[0, 0].legend(loc = 'lower right')
axes[0, 0].grid(True, alpha = 0.3, axis = 'x')

#======================================================================================================================================

# Gráfico 2: Intervalo de Confiança das Previsões
comparacao_sorted = comparacao.sort_values('CodConcelho')
x_pos = np.arange(len(comparacao_sorted))

axes[0, 1].errorbar(x_pos, comparacao_sorted['Y_Previsto'],
                    yerr = [comparacao_sorted['Y_Previsto'] - comparacao_sorted['Y_Previsto_IC_Inferior'],
                          comparacao_sorted['Y_Previsto_IC_Superior'] - comparacao_sorted['Y_Previsto']],
                    fmt = 'o', capsize = 5, capthick = 2, alpha = 0.7, label = 'IC 95%')

axes[0, 1].scatter(x_pos, comparacao_sorted['Y_Real'], color = 'red', s = 100, marker = 'x', linewidths = 2, label = 'Valor Real', zorder = 5)

axes[0, 1].set_xticks(x_pos)

axes[0, 1].set_xticklabels(comparacao_sorted['Concelho'], rotation = 45, ha = 'right')

axes[0, 1].set_ylabel('Utilização Média (Y)')

axes[0, 1].set_title('Intervalos de Confiança 95% das Previsões')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha = 0.3, axis = 'y')

#======================================================================================================================================

# Gráfico 3: Potência Disponível vs Carregadores
axes[1, 0].bar(comparacao_sorted['Concelho'], comparacao_sorted['Potencia_Disponivel_Potencial'], color = 'steelblue', alpha = 0.7, label = 'Potência Disponível (kW)')
ax2 = axes[1, 0].twinx()
ax2.plot(x_pos, comparacao_sorted['N_Carregadores_Estimado'], 'ro-', linewidth = 2, markersize = 8, label = 'Carregadores 22kW')

axes[1, 0].set_xlabel('Concelho')
axes[1, 0].set_ylabel('Potência Disponível (kW)', color = 'steelblue')

ax2.set_ylabel('Nº de Carregadores 22 kW', color = 'red')
axes[1, 0].set_title('Potência Disponível e Capacidade de Carregadores por Concelho')
axes[1, 0].tick_params(axis = 'x', rotation = 45)
axes[1, 0].grid(True, alpha = 0.3, axis = 'y')

#======================================================================================================================================

# Gráfico 4: Amplitude do IC vs Margem de Segurança
scatter = axes[1, 1].scatter(comparacao_sorted['Amplitude_IC'], comparacao_sorted['N_Carregadores_Estimado'], s = 200, alpha = 0.7, c = comparacao_sorted['Score_Viabilidade'], cmap = 'RdYlGn')

for idx, (i, row) in enumerate(comparacao_sorted.iterrows()):
    axes[1, 1].annotate(row['Concelho'][:3], (row['Amplitude_IC'], row['N_Carregadores_Estimado']), fontsize = 9, ha = 'center', va = 'center')

axes[1, 1].set_xlabel('Amplitude do Intervalo de Confiança')
axes[1, 1].set_ylabel('Nº de Carregadores 22 kW Estimados')

axes[1, 1].set_title('Relação: Confiança vs Capacidade de Carregadores')

axes[1, 1].grid(True, alpha = 0.3)

# Adicionar barra de cores para o score de viabilidade
cbar = plt.colorbar(scatter, ax=axes[1, 1])
cbar.set_label('Score Viabilidade', rotation = 270, labelpad = 20)

plt.tight_layout()
plt.show()

## CONCLUSÕES E RECOMENDAÇÕES - ANÁLISE DE VIABILIDADE PARA CARREGADORES DE VE 22 kW

### CRITÉRIOS DE VIABILIDADE UTILIZADOS

#### 1. **INTERVALO DE CONFIANÇA 95%**
- Utilizados para quantificar a incerteza das previsões
- Amplitude do IC indica a precisão da previsão
- Amplitude menor = maior confiabilidade

#### 2. **POTÊNCIA DISPONÍVEL POTENCIAL**
- Considera folga atual + ganho com implementação LED (20% redução ineficiência)
- Permite identificar margem real para novas cargas

#### 3. **NÚMERO DE CARREGADORES 22 kW**
- Cálculo: Potência Disponível / 22 kW
- Representa capacidade realista por concelho

---

### RANKING DE VIABILIDADE

Os concelhos são classificados em função do score de viabilidade calculado com base em:
- Margem de confiança das previsões (amplitude do IC)
- Potência disponível potencial
- Potencial de melhoria com implementação de tecnologia LED

**Distribuição esperada:**

|        Classificação         |     Critério     |            Nº Concelhos             |
|:----------------------------:|:----------------:|:-----------------------------------:|
|     🟢 **MUITO VIÁVEL**      |    Score ≥ 75    | Concelhos com excelentes condições  |
|        🟡 **VIÁVEL**         | 60 ≤ Score < 75  |    Concelhos com boas condições     |
| 🟠 **MODERADAMENTE VIÁVEL**  | 45 ≤ Score < 60  |       Concelhos com potencial       |
|      🔴 **NÃO VIÁVEL**       |    Score < 45    |      Concelhos com limitações       |

---

### RECOMENDAÇÕES PRIORITÁRIAS

#### 1. **IMPLEMENTAÇÃO FASEADA**

As seguintes fases são recomendadas para maximizar a eficiência:

- **Fase 1 (Prioritária):** Concelhos com Score ≥ 75 (ex: Santa Maria da Feira)
  - Maior potencial de libertação de potência
  - Previsões mais confiáveis (IC menor)
  - Menor risco de sobrecarga

- **Fase 2 (Secundária):** Concelhos com 60 ≤ Score < 75
  - Potencial moderado de libertação de potência
  - Boa confiabilidade das previsões

- **Fase 3 (Tertária):** Concelhos com 45 ≤ Score < 60
  - Potencial limitado
  - Implementação depende de validação em campo

#### 2. **JUSTIFICAÇÃO ESTATÍSTICA**

A utilização de **intervalos de confiança 95%** garante que:

- As previsões têm 95% de probabilidade de estarem corretas dentro da amplitude especificada
- Os concelhos com amplitude menor têm previsões mais confiáveis
- A margem de segurança reduz o risco de sobrecarga da rede
- A análise considera a incerteza inerente ao modelo de regressão

#### 3. **PRÓXIMOS PASSOS**

1. **Validação em Campo**
   - Confirmar potência de folga medida
   - Validar dados de carga real da rede

2. **Implementação**
   - Começar pelos concelhos de Fase 1
   - Instalação faseada de carregadores 22 kW

3. **Monitoramento**
   - Implementar monitoramento contínuo pós-instalação
   - Consideração do crescimento futuro de carga
   - Avaliação do impacto real versus previsão

---

### CONCLUSÃO GERAL

A metodologia de intervalos de confiança fornece uma base estatística robusta para identificar os concelhos onde a libertação de potência para carregadores de VE é **estatisticamente mais viável**. Os concelhos classificados como "MUITO VIÁVEIS" ou "VIÁVEIS" apresentam condições ótimas para prosseguir com a implementação de infraestrutura de carregamento de 22 kW, com baixo risco de comprometer a estabilidade da rede elétrica.

---

# Dashboard Interativo – Eficiência Energética e Mobilidade Elétrica

Este dashboard foi desenvolvido com o objetivo de proporcionar uma visualização interativa dos principais indicadores relacionados com a eficiência da iluminação pública e a capacidade da rede elétrica para suportar a integração de carregadores de veículos elétricos (VE).

Através deste painel, é possível analisar, ao nível do concelho, o impacto da modernização da iluminação pública para tecnologia LED na libertação de potência e na viabilidade de instalação de infraestrutura de carregamento.

O dashboard integra as seguintes componentes principais:

Perfis horários de consumo da iluminação pública, comparando o cenário atual com o cenário após modernização para tecnologia LED;
Avaliação da capacidade instalada e da capacidade disponível nos Postos de Transformação de Distribuição (PTD);
Estimativa da potência libertada resultante da substituição de tecnologias ineficientes;
Simulação de cenários de integração de carregadores de veículos elétricos e respetivo impacto na carga da rede;
Representação geográfica dos PTDs, permitindo identificar zonas com potencial para instalação de pontos de carregamento.

Este painel constitui uma ferramenta de apoio à decisão, permitindo explorar de forma intuitiva os resultados obtidos nas análises anteriores e avaliar diferentes cenários energéticos ao nível local.

In [ ]:
# Caminhos
caminho_dataset = Path("data/dataset_final.csv")
caminho_ptd = Path("data/PTD_data.xlsx")

# Dataset final
df_dashboard = pd.read_csv(caminho_dataset)

df_dashboard["Viabilidade_VE"] = np.where(
    df_dashboard["D"] >= 0,
    "Viável",
    "Não Viável"
)

# PTDs
ptd_data = pd.read_excel(caminho_ptd)
ptd_data.columns = ptd_data.columns.str.strip()

print("Dataset carregado com sucesso.")
print(f"Número de concelhos: {len(df_dashboard)}")
display(df_dashboard.head())

In [ ]:
# =========================================
# DROPDOWNS: Distrito -> Concelho
# =========================================

# Lista de distritos (ordenada)
lista_distritos = sorted(df_dashboard["Distrito"].dropna().unique())

# Dropdown de distrito
dropdown_distrito = widgets.Dropdown(
    options=lista_distritos,
    description="",
    layout=widgets.Layout(width="250px")
)

# Dropdown de concelho (atualizado dinamicamente)
dropdown_concelho = widgets.Dropdown(
    description="",
    layout=widgets.Layout(width="250px")
)


# Função para atualizar a lista de concelhos quando muda o distrito
def atualizar_concelhos(change):
    distrito = change["new"]

    lista_concelhos = sorted(
        df_dashboard.loc[df_dashboard["Distrito"] == distrito, "Concelho"]
        .dropna()
        .unique()
    )

    dropdown_concelho.options = lista_concelhos

    if lista_concelhos:
        dropdown_concelho.value = lista_concelhos[0]


# Ligar evento
dropdown_distrito.observe(atualizar_concelhos, names="value")

# Inicializar dropdowns
if lista_distritos:
    dropdown_distrito.value = lista_distritos[0]

In [ ]:
def get_dados_concelho():
    # Obtém os dados do concelho selecionado nos dropdowns
    distrito = dropdown_distrito.value
    concelho = dropdown_concelho.value

    df_filtrado = df_dashboard.loc[
        (df_dashboard["Distrito"] == distrito) &
        (df_dashboard["Concelho"] == concelho)
        ]

    if df_filtrado.empty:
        return None

    return df_filtrado.iloc[0]

In [ ]:
horas = list(range(24))

# Perfil horário normalizado (0–1) da iluminação pública ao longo do dia
# Valores elevados durante a noite e reduzidos durante o dia
perfil_iluminacao = np.array([
    0.75, 0.85, 0.95, 1.00, 1.00, 0.90, 0.60, 0.25,
    0.05, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.05,
    0.20, 0.50, 0.80, 0.95, 1.00, 1.00, 0.90, 0.80
])

assert len(perfil_iluminacao) == 24

In [ ]:
def mostrar_metricas(row):
    estado = row["Viabilidade_VE"]
    emoji = "🟢" if estado == "Viável" else "🔴"
    cor = "green" if estado == "Viável" else "red"

    fator_potencia = 0.9

    capacidade_kw = row["Cap_PTD"] * fator_potencia
    folga_kw = row["PFolga"] * fator_potencia

    potencia_apos_led = row["P_IP_TOTAL"] - row["Ganho_LED"]

    # 🔥 saldo corrigido em kW
    saldo_kw = folga_kw + row["Ganho_LED"] - row["PVE"]

    display(Markdown(f"""
## {row['Concelho']} ({row['Distrito']})

### ⚡ Energia e Infraestrutura
- **Potência total:** {row['P_IP_TOTAL']:.2f} kW
- **Potência ineficiente:** {row['P_IP_Inef']:.2f} kW
- **Potência libertada (LED) pelas medidas de eficiência:** {row['Ganho_LED']:.2f} kW
- **Potência após LED:** {potencia_apos_led:.2f} kW
- **Capacidade PTD:** {capacidade_kw:.2f} kW
- **Utilização média:** {row['Util_Media']:.2%}
- **Folga da rede:** {folga_kw:.2f} kW

### 🚗 Mobilidade Elétrica
- **Carga VE estimada:** {row['PVE']:.2f} kW

---

## 🧮 Resultado Final (cenário com LED)

### **Saldo (D): {saldo_kw:.2f} kW**

### <span style="color:{cor}; font-size:18px;"><b>{emoji} {estado}</b></span>
"""))

In [ ]:
def plot_perfil_horario(row):
    # Estimar consumo antes e depois da modernização LED
    consumo_antes = row["P_IP_TOTAL"] * perfil_iluminacao

    potencia_depois = max(row["P_IP_TOTAL"] - row["Ganho_LED"], 0)
    consumo_depois = potencia_depois * perfil_iluminacao

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=horas,
        y=consumo_antes,
        mode="lines+markers",
        name="Antes (tecnologia convencional)"
    ))

    fig.add_trace(go.Scatter(
        x=horas,
        y=consumo_depois,
        mode="lines+markers",
        name="Depois (tecnologia LED)"
    ))

    fig.update_layout(
        title=f"Perfil horário de consumo — {row['Concelho']}",
        xaxis_title="Hora do dia",
        yaxis_title="Potência estimada (kW)",
        height=450
    )

    fig.show()

In [ ]:
def plot_capacidade_viabilidade(row):
    fator_potencia = 0.9

    # Conversões para kW
    folga_kw = row["PFolga"] * fator_potencia
    ganho_led = row["Ganho_LED"]
    carga_ve = row["PVE"]

    # 🔥 saldo corrigido (não usar row["D"])
    saldo = folga_kw + ganho_led - carga_ve

    categorias = [
        "Folga atual",
        "Ganho LED",
        "Carga VE",
        "Saldo final"
    ]

    valores = [
        folga_kw,
        ganho_led,
        carga_ve,
        saldo
    ]

    cores = [
        "#B0BEC5",  # folga
        "#66BB6A",  # ganho LED
        "#EF5350",  # carga VE
        "#2E8B57" if saldo >= 0 else "#C0392B"  # saldo
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=categorias,
                y=valores,
                text=[f"{v:.2f}" for v in valores],
                textposition="outside",
                marker_color=cores
            )
        ]
    )

    fig.update_layout(
        title=f"Viabilidade da rede após modernização LED — {row['Concelho']}",
        yaxis_title="Potência (kW)",
        template="plotly_white",
        height=460,
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
        yaxis=dict(showgrid=True, gridcolor="#EAEAEA"),
        xaxis=dict(showgrid=False)
    )

    fig.show()

In [ ]:
def plot_mapa_ptd(row):
    codigo = row["CodDistritoConcelho"]

    df_ptd = ptd_data[ptd_data["CodDistritoConcelho"] == codigo].copy()

    if df_ptd.empty:
        print("Sem PTDs para este concelho.")
        return

    coords = df_ptd["Coordenadas Geográficas"].astype(str).str.replace(" ", "").str.split(",", expand=True)
    df_ptd["lat"] = pd.to_numeric(coords[0], errors="coerce")
    df_ptd["lon"] = pd.to_numeric(coords[1], errors="coerce")
    df_ptd = df_ptd.dropna(subset=["lat", "lon"])

    if df_ptd.empty:
        print("Sem coordenadas válidas para este concelho.")
        return

    fig = px.scatter_map(
        df_ptd,
        lat="lat",
        lon="lon",
        hover_name="Tipo Construtivo" if "Tipo Construtivo" in df_ptd.columns else None,
        hover_data={
            "Código de Instalação": "Código de Instalação" in df_ptd.columns,
            "Potência instalada [kVA]": "Potência instalada [kVA]" in df_ptd.columns,
            "Nível de Utilização [%]": "Nível de Utilização [%]" in df_ptd.columns,
            "Concelho": "Concelho" in df_ptd.columns,
            "lat": False,
            "lon": False
        },
        zoom=11,
        height=500,
        title=f"Mapa dos PTDs — {row['Concelho']}",
        color_discrete_sequence=["#2E8B57"]
    )

    fig.update_layout(
        map_style="open-street-map",
        margin=dict(l=0, r=0, t=50, b=0)
    )

    fig.show()

In [ ]:
output_dashboard = widgets.Output()

In [ ]:
def atualizar_dashboard(*args):
    with output_dashboard:
        output_dashboard.clear_output()

        row = get_dados_concelho()
        if row is None:
            print("Nenhum concelho selecionado.")
            return

        # KPIs
        display(Markdown(
            "#### Este painel resume os principais indicadores energéticos e de capacidade da rede para o concelho selecionado."
        ))
        mostrar_metricas(row)
        display(Markdown("---"))

        # Viabilidade
        display(Markdown(
            "#### Este gráfico apresenta a folga atual da rede, o ganho estimado com a modernização LED, a carga dos veículos elétricos e o saldo final de viabilidade."
        ))
        plot_capacidade_viabilidade(row)
        display(Markdown("---"))

        # Perfil horário
        display(Markdown(
            "#### O perfil horário mostra a variação estimada do consumo de iluminação pública ao longo do dia, antes e depois da modernização para tecnologia LED."
        ))
        plot_perfil_horario(row)
        display(Markdown("---"))

        # Mapa
        display(Markdown(
            "#### O mapa seguinte mostra a localização dos PTDs do concelho selecionado, identificando a infraestrutura onde poderão ser analisados potenciais pontos de carregamento."
        ))
        plot_mapa_ptd(row)

In [ ]:
dropdown_distrito.observe(atualizar_dashboard, names="value")
dropdown_concelho.observe(atualizar_dashboard, names="value")

display(widgets.HBox([dropdown_distrito, dropdown_concelho]))
display(output_dashboard)

atualizar_dashboard()